# World GDP Growth - Exploratory Data Analysis

This notebook performs exploratory data analysis on the World GDP Growth dataset (1980-2024).

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Data Loading and Preprocessing

In [ ]:
# Load data
import sys
sys.path.append('../')

from utils.data_loader import load_and_clean_data

df = load_and_clean_data('../data/world_gdp_data.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

In [ ]:
# Data info
df.info()

In [ ]:
# Basic statistics
df['gdp_growth'].describe()

## 2. Global Trends Analysis

In [ ]:
# Global average GDP growth over time
global_trend = df.groupby('year')['gdp_growth'].mean().reset_index()

fig = px.line(global_trend, x='year', y='gdp_growth',
              title='Global Average GDP Growth (1980-2024)',
              labels={'gdp_growth': 'GDP Growth (%)', 'year': 'Year'})
fig.add_hline(y=0, line_dash="dash", line_color="red", annotation_text="Zero Growth")
fig.update_layout(template='plotly_white', height=500)
fig.show()

In [ ]:
# Distribution of GDP growth
fig = px.histogram(df, x='gdp_growth', nbins=50,
                   title='Distribution of GDP Growth Rates',
                   labels={'gdp_growth': 'GDP Growth (%)'})
fig.add_vline(x=df['gdp_growth'].mean(), line_dash="dash", 
              line_color="red", annotation_text=f"Mean: {df['gdp_growth'].mean():.2f}%")
fig.update_layout(template='plotly_white', height=500)
fig.show()

## 3. Country-Level Analysis

In [ ]:
# Top 10 countries by average GDP growth
country_avg = df.groupby('country_name')['gdp_growth'].mean().sort_values(ascending=False).head(10)

fig = px.bar(x=country_avg.values, y=country_avg.index, orientation='h',
             title='Top 10 Countries by Average GDP Growth',
             labels={'x': 'Average GDP Growth (%)', 'y': 'Country'})
fig.update_layout(template='plotly_white', height=500, yaxis={'categoryorder':'total ascending'})
fig.show()

In [ ]:
# GDP growth trends for major economies
major_economies = ['United States', 'China', 'India', 'Japan', 'Germany', 'United Kingdom']
major_df = df[df['country_name'].isin(major_economies)]

fig = px.line(major_df, x='year', y='gdp_growth', color='country_name',
              title='GDP Growth Trends - Major Economies',
              labels={'gdp_growth': 'GDP Growth (%)', 'year': 'Year', 'country_name': 'Country'})
fig.update_layout(template='plotly_white', height=600, hovermode='x unified')
fig.show()

## 4. Regional Analysis

In [ ]:
# Regional comparison boxplot
fig = px.box(df, x='region', y='gdp_growth', color='region',
             title='GDP Growth Distribution by Region',
             labels={'gdp_growth': 'GDP Growth (%)', 'region': 'Region'})
fig.update_layout(template='plotly_white', height=500, showlegend=False)
fig.show()

In [ ]:
# Regional trends over time
regional_trend = df.groupby(['year', 'region'])['gdp_growth'].mean().reset_index()

fig = px.line(regional_trend, x='year', y='gdp_growth', color='region',
              title='Regional GDP Growth Trends',
              labels={'gdp_growth': 'Average GDP Growth (%)', 'year': 'Year', 'region': 'Region'})
fig.update_layout(template='plotly_white', height=600)
fig.show()

## 5. Crisis Analysis

In [ ]:
# Key economic events
crisis_years = {
    2008: '2008 Financial Crisis',
    2009: '2009 Global Recession',
    2020: '2020 COVID-19 Pandemic'
}

for year, event in crisis_years.items():
    crisis_data = df[df['year'] == year]['gdp_growth']
    print(f"\n{event} ({year})")
    print(f"  Average GDP Growth: {crisis_data.mean():.2f}%")
    print(f"  Countries with negative growth: {(crisis_data < 0).sum()} / {len(crisis_data)}")
    print(f"  Worst performer: {crisis_data.min():.2f}%")

In [ ]:
# Visualize crisis impact
crisis_df = df[df['year'].isin([2007, 2008, 2009, 2010, 2019, 2020, 2021, 2022])]

fig = px.box(crisis_df, x='year', y='gdp_growth',
             title='GDP Growth During Crisis Periods',
             labels={'gdp_growth': 'GDP Growth (%)', 'year': 'Year'})
fig.add_hline(y=0, line_dash="dash", line_color="red")
fig.update_layout(template='plotly_white', height=500)
fig.show()

## 6. Volatility Analysis

In [ ]:
# Calculate volatility (standard deviation) for each country
volatility = df.groupby('country_name')['gdp_growth'].std().sort_values(ascending=False)

print("Top 10 Most Volatile Economies:")
print(volatility.head(10))

print("\nTop 10 Most Stable Economies:")
print(volatility.tail(10))

In [ ]:
# Scatter plot: Average growth vs Volatility
country_stats = df.groupby('country_name').agg({
    'gdp_growth': ['mean', 'std']
}).reset_index()
country_stats.columns = ['country', 'avg_growth', 'volatility']
country_stats['region'] = country_stats['country'].map(
    df.groupby('country_name')['region'].first().to_dict()
)

fig = px.scatter(country_stats, x='avg_growth', y='volatility', 
                 color='region', hover_data=['country'],
                 title='GDP Growth: Average vs Volatility',
                 labels={'avg_growth': 'Average GDP Growth (%)', 
                        'volatility': 'Volatility (Std Dev)'})
fig.update_layout(template='plotly_white', height=600)
fig.show()

## 7. Correlation Analysis

In [ ]:
# Create pivot table for correlation analysis
pivot_df = df.pivot_table(index='year', columns='region', values='gdp_growth', aggfunc='mean')

# Calculate correlation
corr_matrix = pivot_df.corr()

# Plot heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text:.2f}',
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title='Regional GDP Growth Correlation Matrix',
    template='plotly_white',
    height=600
)
fig.show()

## 8. Key Insights Summary

In [ ]:
print("=" * 80)
print("KEY INSIGHTS FROM EXPLORATORY DATA ANALYSIS")
print("=" * 80)

print(f"\n1. OVERALL STATISTICS")
print(f"   - Global average GDP growth: {df['gdp_growth'].mean():.2f}%")
print(f"   - Median GDP growth: {df['gdp_growth'].median():.2f}%")
print(f"   - Standard deviation: {df['gdp_growth'].std():.2f}%")
print(f"   - Range: {df['gdp_growth'].min():.2f}% to {df['gdp_growth'].max():.2f}%")

print(f"\n2. TOP PERFORMERS")
top_3 = df.groupby('country_name')['gdp_growth'].mean().sort_values(ascending=False).head(3)
for i, (country, growth) in enumerate(top_3.items(), 1):
    print(f"   {i}. {country}: {growth:.2f}%")

print(f"\n3. REGIONAL PERFORMANCE")
regional_avg = df.groupby('region')['gdp_growth'].mean().sort_values(ascending=False)
for region, growth in regional_avg.items():
    print(f"   - {region}: {growth:.2f}%")

print(f"\n4. HISTORICAL MILESTONES")
yearly_avg = df.groupby('year')['gdp_growth'].mean()
print(f"   - Best year: {yearly_avg.idxmax()} ({yearly_avg.max():.2f}%)")
print(f"   - Worst year: {yearly_avg.idxmin()} ({yearly_avg.min():.2f}%)")

print(f"\n5. RECENT TRENDS (2020-2024)")
recent = df[df['year'] >= 2020].groupby('year')['gdp_growth'].mean()
for year, growth in recent.items():
    print(f"   - {year}: {growth:.2f}%")

print("\n" + "=" * 80)

## 9. Save Processed Data

In [ ]:
# Save processed data for use in the dashboard
df.to_csv('../data/processed_gdp_data.csv', index=False)
print("Processed data saved to '../data/processed_gdp_data.csv'")